In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


In [2]:
train = pd.read_csv(
    "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip",
    header=0,
    delimiter="\t",
    quoting=3
)

train.shape

(25000, 3)

In [3]:
train.head()

,id,sentiment,review
0,"""5814_8""",1,"""With all this stuff going down at the moment ..."
1,"""2381_9""",1,"""\""The Classic War of the Worlds\"" by Timothy ..."
2,"""7759_3""",0,"""The film starts with a manager (Nicholas Bell..."
3,"""3630_4""",0,"""It must be assumed that those who praised thi..."
4,"""9495_8""",1,"""Superbly trashy and wondrously unpretentious ..."


In [4]:
print("The first review is:")
print(train["review"][0])

from bs4 import BeautifulSoup

review_text = BeautifulSoup(
    train["review"][0],
    "html.parser"
).get_text()

print(review_text)

The first review is:
"With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.<br /><br />Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.<br /><br />The actual feature film bi

In [5]:
from bs4 import BeautifulSoup

review_text = BeautifulSoup(
    train["review"][0],
    "html.parser"
).get_text()

print(review_text)

"With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.The actual feature film bit when it finally starts is only on for 20 mi

In [6]:
import re

review_text = re.sub("[^a-zA-Z]", " ", review_text)

print(review_text[:500])

 With all this stuff going down at the moment with MJ i ve started listening to his music  watching the odd documentary here and there  watched The Wiz and watched Moonwalker again  Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent  Moonwalker is part biography  part feature film which i remember going to see at the cinema when it was originally released  Some of it has subtle mess


In [7]:
words = review_text.lower().split()

print(words[:50])

['with', 'all', 'this', 'stuff', 'going', 'down', 'at', 'the', 'moment', 'with', 'mj', 'i', 've', 'started', 'listening', 'to', 'his', 'music', 'watching', 'the', 'odd', 'documentary', 'here', 'and', 'there', 'watched', 'the', 'wiz', 'and', 'watched', 'moonwalker', 'again', 'maybe', 'i', 'just', 'want', 'to', 'get', 'a', 'certain', 'insight', 'into', 'this', 'guy', 'who', 'i', 'thought', 'was', 'really', 'cool']


In [8]:
from nltk.corpus import stopwords

stops = set(stopwords.words("english"))

print("Stopword count:", len(stops))

Stopword count: 198


In [9]:
words_without_stopwords = [w for w in words if not w in stops]

print("Before:")
print(words[:50])

print("\nAfter:")
print(words_without_stopwords[:50])

Before:
['with', 'all', 'this', 'stuff', 'going', 'down', 'at', 'the', 'moment', 'with', 'mj', 'i', 've', 'started', 'listening', 'to', 'his', 'music', 'watching', 'the', 'odd', 'documentary', 'here', 'and', 'there', 'watched', 'the', 'wiz', 'and', 'watched', 'moonwalker', 'again', 'maybe', 'i', 'just', 'want', 'to', 'get', 'a', 'certain', 'insight', 'into', 'this', 'guy', 'who', 'i', 'thought', 'was', 'really', 'cool']

After:
['stuff', 'going', 'moment', 'mj', 'started', 'listening', 'music', 'watching', 'odd', 'documentary', 'watched', 'wiz', 'watched', 'moonwalker', 'maybe', 'want', 'get', 'certain', 'insight', 'guy', 'thought', 'really', 'cool', 'eighties', 'maybe', 'make', 'mind', 'whether', 'guilty', 'innocent', 'moonwalker', 'part', 'biography', 'part', 'feature', 'film', 'remember', 'going', 'see', 'cinema', 'originally', 'released', 'subtle', 'messages', 'mj', 'feeling', 'towards', 'press', 'also', 'obvious']


In [10]:
import re
from bs4 import BeautifulSoup
from nltk.corpus import stopwords


def review_to_wordlist(review, remove_stopwords=False):
    # 1. Remove HTML
    review_text = BeautifulSoup(review, "html.parser").get_text()

    # 2. Remove non-letters
    review_text = re.sub("[^a-zA-Z]", " ", review_text)

    # 3. Convert words to lower case and split them
    words = review_text.lower().split()

    # 4. Optionally remove stop words
    if remove_stopwords:
        stops = set(stopwords.words("english"))
        words = [w for w in words if w not in stops]

    # 5. Return a list of words
    return words

In [11]:
clean_words = review_to_wordlist(
    train["review"][0],
    remove_stopwords=True
)

print(clean_words[:50])

['stuff', 'going', 'moment', 'mj', 'started', 'listening', 'music', 'watching', 'odd', 'documentary', 'watched', 'wiz', 'watched', 'moonwalker', 'maybe', 'want', 'get', 'certain', 'insight', 'guy', 'thought', 'really', 'cool', 'eighties', 'maybe', 'make', 'mind', 'whether', 'guilty', 'innocent', 'moonwalker', 'part', 'biography', 'part', 'feature', 'film', 'remember', 'going', 'see', 'cinema', 'originally', 'released', 'subtle', 'messages', 'mj', 'feeling', 'towards', 'press', 'also', 'obvious']


In [12]:
test_clean_train_reviews = []

In [13]:
print(train.shape)
print(review_to_wordlist)

(25000, 3)
<function review_to_wordlist at 0x7c4bf96efba0>


In [14]:
print("Cleaning and parsing the first 5 training reviews...\n")

for i in range(0, 5):
    test_clean_train_reviews.append(
        " ".join(
            review_to_wordlist(
                train["review"][i],
                True
            )
        )
    )

Cleaning and parsing the first 5 training reviews...



In [15]:
clean_train_reviews = []

In [16]:
clean_train_reviews = []

print("Cleaning and parsing the training set movie reviews...\n")

for i in range(0, len(train["review"])):
    clean_train_reviews.append(
        " ".join(
            review_to_wordlist(
                train["review"][i],
                True
            )
        )
    )

Cleaning and parsing the training set movie reviews...



In [17]:
print(len(clean_train_reviews))

25000


In [18]:
from sklearn.feature_extraction.text import CountVectorizer

In [19]:
print("Creating the bag of words...\n")

vectorizer = CountVectorizer(
    analyzer="word",
    tokenizer=None,
    preprocessor=None,
    stop_words=None,
    max_features=5000
)

Creating the bag of words...



In [20]:
train_data_features = vectorizer.fit_transform(clean_train_reviews)

In [21]:
print(train_data_features.shape)

(25000, 5000)


In [22]:
vocab = vectorizer.get_feature_names_out()

print("Vocabulary size:", len(vocab))
print(vocab[:50])

Vocabulary size: 5000
['abandoned' 'abc' 'abilities' 'ability' 'able' 'abraham' 'absence'
 'absent' 'absolute' 'absolutely' 'absurd' 'abuse' 'abusive' 'abysmal'
 'academy' 'accent' 'accents' 'accept' 'acceptable' 'accepted' 'access'
 'accident' 'accidentally' 'accompanied' 'accomplished' 'according'
 'account' 'accuracy' 'accurate' 'accused' 'achieve' 'achieved'
 'achievement' 'acid' 'across' 'act' 'acted' 'acting' 'action' 'actions'
 'activities' 'actor' 'actors' 'actress' 'actresses' 'acts' 'actual'
 'actually' 'ad' 'adam']


In [23]:
first_review_vector = train_data_features[0].toarray()[0]

nonzero_indices = np.nonzero(first_review_vector)[0]

print("Number of vocabulary words appearing in the first review:",
      len(nonzero_indices))

for index in nonzero_indices[:30]:
    print(vocab[index], ":", first_review_vector[index])

Number of vocabulary words appearing in the first review: 146
actual : 1
alone : 1
also : 2
another : 1
anyway : 1
attention : 1
away : 1
bad : 3
behind : 1
beyond : 1
biography : 1
bit : 1
boring : 1
bottom : 1
buddy : 1
bunch : 1
call : 1
came : 1
car : 1
certain : 1
character : 1
cinema : 1
closed : 1
complex : 1
convincing : 1
cool : 2
course : 1
criminal : 1
dance : 1
dead : 1


In [24]:
from sklearn.ensemble import RandomForestClassifier

print("Training the random forest (this may take a while)...")

forest = RandomForestClassifier(
    n_estimators=100
)

forest = forest.fit(
    train_data_features,
    train["sentiment"]
)

Training the random forest (this may take a while)...


In [25]:
test = pd.read_csv(
    "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip",
    header=0,
    delimiter="\t",
    quoting=3
)

print(test.shape)
test.head()

(25000, 2)


,id,review
0,"""12311_10""","""Naturally in a film who's main themes are of ..."
1,"""8348_2""","""This movie is a disaster within a disaster fi..."
2,"""5828_4""","""All in all, this is a movie for kids. We saw ..."
3,"""7186_2""","""Afraid of the Dark left me with the impressio..."
4,"""12128_7""","""A very accurate depiction of small time mob l..."


In [26]:
clean_test_reviews = []

print("Cleaning and parsing the test set movie reviews...\n")

for i in range(0, len(test["review"])):
    clean_test_reviews.append(
        " ".join(
            review_to_wordlist(
                test["review"][i],
                True
            )
        )
    )

Cleaning and parsing the test set movie reviews...



In [27]:
test_data_features = vectorizer.transform(clean_test_reviews)

print(test_data_features.shape)

(25000, 5000)


In [28]:
print("Predicting test labels...\n")

result = forest.predict(test_data_features)

print(result[:20])
print("Number of predictions:", len(result))

Predicting test labels...

[1 0 1 1 1 0 0 0 0 1 0 1 1 1 0 1 0 0 1 0]
Number of predictions: 25000


In [29]:
output = pd.DataFrame(
    data={
        "id": test["id"],
        "sentiment": result
    }
)

output.to_csv(
    "/kaggle/working/Bag_of_Words_model.csv",
    index=False,
    quoting=3
)

print("Wrote results to Bag_of_Words_model.csv")

Wrote results to Bag_of_Words_model.csv


In [30]:
output.head()

,id,sentiment
0,"""12311_10""",1
1,"""8348_2""",0
2,"""5828_4""",1
3,"""7186_2""",1
4,"""12128_7""",1
